# BioGym Arena: version 2 SFT + RL and transfer evaluation
Generate verified train-only tool trajectories, warm-start Qwen3-0.6B with LoRA SFT, then apply GRPO in the version 2 BioGym environment. Evaluate frozen before-RL and after-RL adapters on both held-out test and TransferArena splits in separate GPU processes. This is **not Sarvam 30B training**. Use a GPU runtime and review `docs/rl-training.md` for provenance and limits.


In [ ]:
import torch, subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before training'


In [ ]:
!git clone --depth 1 --branch feat/application-v1 https://github.com/Ajey95/biogym-arena.git /content/biogym-arena-v2
%cd /content/biogym-arena-v2
# Colab's preinstalled torchao 0.10 is incompatible with current PEFT.
%pip uninstall -y torchao
%pip install -q -e 'backend[train]'


In [ ]:
# Use a fresh Python process so the editable install is on sys.path.
!python -c "from biogym.rl_suite import SUITE_HASH, split_tasks; print('Suite:', SUITE_HASH, {s: len(split_tasks(s)) for s in ('train','test','transfer')})"
!python -m biogym.rl_sft --output /content/biogym-v2-sft --model Qwen/Qwen3-0.6B --epochs 1
!python -m biogym.rl_train --output /content/biogym-v2-run --model Qwen/Qwen3-0.6B --warm-adapter /content/biogym-v2-sft/adapter --max-steps 32


In [ ]:
import subprocess
from pathlib import Path
run_dir = '/content/biogym-v2-run'
for split in ('test', 'transfer'):
    for adapter in ('pre', 'post'):
        output = f'/content/biogym-v2-{split}-{adapter}'
        log = f'{output}.log'
        with open(log, 'w') as stream:
            subprocess.run(['python', '-m', 'biogym.rl_eval', '--run-dir', run_dir, '--adapter', adapter, '--split', split, '--output', output], stdout=stream, stderr=subprocess.STDOUT, check=True)
        print(split, adapter, 'complete:', log)


In [ ]:
import json, subprocess
from pathlib import Path
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
for split in ('test', 'transfer'):
    prefix = f'/content/biogym-v2-{split}'
    subprocess.run(['python', '-m', 'biogym.rl_compare', '--run-dir', '/content/biogym-v2-run', '--pre-eval', f'{prefix}-pre', '--post-eval', f'{prefix}-post', '--pre-log', f'{prefix}-pre.log', '--post-log', f'{prefix}-post.log', '--repo-dir', '/content/biogym-arena-v2', '--evaluation-code-commit', commit, '--output', f'{prefix}.zip'], check=True)
    report = json.loads(Path(f'{prefix}.json').read_text())
    print(split, report['before_successes'], '->', report['after_successes'], '/', report['attempts_per_policy'], 'artifact:', f'{prefix}.zip')
